# LLM Fine-Tuning — End-to-End Tutorial

Covers the full pipeline from raw tensors to a deployed fine-tuned LLM.

| Section | Topic | Kernel |
|---------|-------|--------|
| 1 | PyTorch basics | `.venv` (Unsloth+SFT) |
| 2 | HuggingFace Datasets | `.venv` |
| 3 | HuggingFace Transformers + Trainer | `.venv` |
| 4 | Causal LM fine-tuning (GPT-2) | `.venv` |
| 5 | LoRA with PEFT | `.venv` |
| 6 | Unsloth SFT workflow | `.venv` |
| 7 | DPO | `.venv-rl` (DPO/ORPO/GRPO) |
| 8 | ORPO | `.venv-rl` |
| 9 | GRPO | `.venv-rl` |

> **Switch kernel for sections 7-9:** `Ctrl+Shift+P` → *Notebook: Select Kernel* → `Python (.venv-rl — DPO/ORPO/GRPO)`

---
## Section 1 — PyTorch Basics

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__} | device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Tensors
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]]).to(device)
print('x:', x)
print('x @ xT:', x @ x.T)

# Autograd
a = torch.tensor(3.0, requires_grad=True)
b = a ** 2 + 2 * a + 1
b.backward()
print(f'\nb=a^2+2a+1 at a=3 -> b={b.item():.1f}, db/da={a.grad.item():.1f}  (expected 8)')

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2))
    def forward(self, x):
        return self.net(x)

class ToyDataset(Dataset):
    def __init__(self, n=200):
        self.X = torch.randn(n, 4)
        self.y = (self.X[:, 0] > 0).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

model     = TinyMLP().to(device)
loader    = DataLoader(ToyDataset(), batch_size=32, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f'epoch {epoch+1}  loss={total/len(loader):.4f}')

---
## Section 2 — HuggingFace Datasets

In [ ]:
from datasets import Dataset, DatasetDict, concatenate_datasets

raw = {
    'text':  ['LoRA reduces trainable parameters.',
              'Unsloth speeds up fine-tuning 2x.',
              'DPO aligns models with preferences.',
              'GRPO uses group reward signals.',
              'QLoRA combines 4-bit quantization with LoRA.'],
    'label': [1, 1, 1, 1, 1],
}
ds = Dataset.from_dict(raw)
print(ds)
print(ds[0])

In [ ]:
# map, filter, split, set_format
ds = ds.map(lambda x: {'word_count': len(x['text'].split())})
ds = ds.filter(lambda x: x['word_count'] >= 4)
print('after filter:', len(ds), 'rows')

splits = ds.train_test_split(test_size=0.4, seed=42)
print(splits)

splits.set_format('torch', columns=['label'])
print('label tensor:', splits['train'][0]['label'])

In [ ]:
# Load from Hub (requires internet)
# from datasets import load_dataset
# alpaca   = load_dataset('tatsu-lab/alpaca',          split='train')
# gsm8k    = load_dataset('openai/gsm8k', 'main',     split='train')
# finetome = load_dataset('mlabonne/FineTome-100k',    split='train')
# orpo_mix = load_dataset('mlabonne/orpo-dpo-mix-40k', split='train')
print('Uncomment above to load real datasets (internet required)')

---
## Section 3 — HuggingFace Transformers + Trainer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset
import tempfile

MODEL = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(MODEL)

enc = tok('HuggingFace Transformers are great!', return_tensors='pt')
print('input_ids shape:', enc['input_ids'].shape)
print('decoded:', tok.decode(enc['input_ids'][0]))

In [ ]:
texts  = ['I love NLP!', 'I hate bugs.', 'Models are fun.', 'Training is slow.'] * 8
labels = [1, 0, 1, 0] * 8

raw_ds = Dataset.from_dict({'text': texts, 'label': labels})
splits  = raw_ds.train_test_split(test_size=0.2, seed=42)

def tokenize(batch):
    return tok(batch['text'], truncation=True, max_length=32)

tok_ds  = splits.map(tokenize, batched=True, remove_columns=['text'])
tok_ds.set_format('torch')
collator = DataCollatorWithPadding(tok)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

with tempfile.TemporaryDirectory() as tmp:
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=tmp, num_train_epochs=2,
            per_device_train_batch_size=8,
            eval_strategy='epoch',   # correct (not evaluation_strategy)
            save_strategy='no', report_to='none',
        ),
        train_dataset=tok_ds['train'],
        eval_dataset=tok_ds['test'],
        processing_class=tok,        # modern API
        data_collator=collator,
    )
    result = trainer.train()
    print(f'loss: {result.training_loss:.4f}')
    # eval metrics already computed by eval_strategy=epoch
    eval_metrics = {k:v for d in trainer.state.log_history for k,v in d.items() if "eval_" in k}
    print(eval_metrics)


---
## Section 4 — Causal LM Fine-Tuning (GPT-2)

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments,
)
from datasets import Dataset
import tempfile, torch

MODEL = 'gpt2'
tok   = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL)
model.config.pad_token_id = tok.eos_token_id

corpus = [
    'The transformer architecture revolutionized natural language processing.',
    'Attention mechanisms allow models to focus on relevant parts of the input.',
    'Fine-tuning adapts a pre-trained model to a specific downstream task.',
    'LoRA reduces trainable parameters by injecting low-rank matrices.',
    'Supervised fine-tuning teaches a model to follow instructions.',
] * 12

raw_ds = Dataset.from_dict({'text': corpus})
splits  = raw_ds.train_test_split(test_size=0.2, seed=42)

def tokenize(batch):
    return tok(batch['text'], truncation=True, max_length=64, padding='max_length')

tok_ds = splits.map(tokenize, batched=True, remove_columns=['text'])
tok_ds.set_format('torch')

# mlm=False -> causal LM; labels auto-set to input_ids
collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

with tempfile.TemporaryDirectory() as tmp:
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=tmp, num_train_epochs=2,
            per_device_train_batch_size=8, eval_strategy='epoch',
            save_strategy='no', report_to='none',
            fp16=torch.cuda.is_available(),
        ),
        train_dataset=tok_ds['train'],
        eval_dataset=tok_ds['test'],
        processing_class=tok,
        data_collator=collator,
    )
    result = trainer.train()
    print(f'loss: {result.training_loss:.4f}')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval().to(device)
inputs = tok('The transformer architecture', return_tensors='pt').to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=40, do_sample=True,
                         temperature=0.8, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=True))

---
## Section 5 — LoRA with PEFT

In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from transformers import AutoModelForCausalLM
import tempfile

base = AutoModelForCausalLM.from_pretrained('distilgpt2')

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['c_attn'],
    task_type=TaskType.CAUSAL_LM,
    bias='none',
)
peft_model = get_peft_model(base, lora_cfg)
peft_model.print_trainable_parameters()

# Save adapters only, then reload
with tempfile.TemporaryDirectory() as tmp:
    peft_model.save_pretrained(tmp)
    base2    = AutoModelForCausalLM.from_pretrained('distilgpt2')
    reloaded = PeftModel.from_pretrained(base2, tmp)
    merged   = reloaded.merge_and_unload()
    print('Merged model params:', sum(p.numel() for p in merged.parameters()))

---
## Section 6 — Unsloth SFT Workflow

Uses `FastLanguageModel` for 4-bit QLoRA + fused kernels (2x faster, 60% less VRAM).

In [ ]:
try:
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template, train_on_responses_only
    import torch
    UNSLOTH = True
    print(f'Unsloth ready | GPU: {torch.cuda.get_device_name(0)}')
except ImportError as e:
    UNSLOTH = False
    print(f'Unsloth not available: {e}')

In [ ]:
if UNSLOTH:
    # Step 1: 4-bit quantized model
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name='unsloth/Meta-Llama-3.1-8B-Instruct',
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )

    # Step 2: LoRA adapters
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj',
                        'gate_proj','up_proj','down_proj'],
        lora_alpha=16, lora_dropout=0.0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=42,
    )
    model.print_trainable_parameters()
else:
    print('Skipped')

In [ ]:
if UNSLOTH:
    from datasets import load_dataset
    tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')

    def format_prompt(examples):
        return {'text': [
            tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
            for c in examples['conversations']
        ]}

    dataset = load_dataset('mlabonne/FineTome-100k', split='train')
    dataset = dataset.map(format_prompt, batched=True)
    print('dataset size:', len(dataset))

In [ ]:
if UNSLOTH:
    from trl import SFTTrainer
    from transformers import TrainingArguments, DataCollatorForSeq2Seq

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field='text', max_seq_length=2048,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=5, max_steps=60,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=1, optim='adamw_8bit',
            weight_decay=0.01, lr_scheduler_type='linear',
            seed=42, output_dir='outputs', report_to='none',
        ),
    )
    trainer = train_on_responses_only(
        trainer,
        instruction_part='<|start_header_id|>user<|end_header_id|>\n\n',
        response_part='<|start_header_id|>assistant<|end_header_id|>\n\n',
    )
    stats = trainer.train()
    print(f'loss: {stats.training_loss:.4f}')

In [ ]:
if UNSLOTH:
    from transformers import TextStreamer
    FastLanguageModel.for_inference(model)

    messages = [{'role': 'user', 'content': 'Explain LoRA in one sentence.'}]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True
    ).to('cuda')
    _ = model.generate(input_ids=inputs,
                       streamer=TextStreamer(tokenizer, skip_prompt=True),
                       max_new_tokens=128, temperature=0.7, top_p=0.9)

In [ ]:
if UNSLOTH:
    model.save_pretrained('lora_model')       # Option A: adapters only
    tokenizer.save_pretrained('lora_model')

    # model.save_pretrained_merged('model_merged', tokenizer, save_method='merged_16bit')
    # model.save_pretrained_gguf('model_gguf', tokenizer, quantization_method='q4_k_m')
    print('Saved to lora_model/')

---
## Section 7 — DPO (Direct Preference Optimization)

> **Switch kernel to `.venv-rl`** — `Ctrl+Shift+P` → *Notebook: Select Kernel* → `Python (.venv-rl — DPO/ORPO/GRPO)`

DPO replaces the RL loop with a classification objective on `(prompt, chosen, rejected)` triples. No reward model needed.

In [ ]:
import torch, tempfile
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer, DPOConfig

print(f'trl: {__import__("trl").__version__}  |  cuda: {torch.cuda.is_available()}')

MODEL = 'gpt2'
tok   = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL)
model.config.pad_token_id = tok.eos_token_id

data = {
    'prompt':   ['What is LoRA?', 'Explain fine-tuning.',
                 'What is RLHF?', 'What is overfitting?',
                 'Define a transformer.', 'What is attention?'] * 6,
    'chosen':   [
        'LoRA adds small trainable rank-decomposition matrices to frozen weights.',
        'Fine-tuning continues training a pre-trained model on task-specific data.',
        'RLHF trains models using human preference feedback via a reward model.',
        'Overfitting is when a model memorises training data and fails to generalise.',
        'A transformer uses self-attention to model relationships between all tokens.',
        'Attention weights how much each token should influence other tokens.',
    ] * 6,
    'rejected': [
        'LoRA makes models faster.', 'Fine-tuning is training from scratch.',
        'RLHF uses robots.', 'Overfitting means the model is very accurate.',
        'A transformer is a type of RNN.', 'Attention is just a linear layer.',
    ] * 6,
}
ds = Dataset.from_dict(data).train_test_split(test_size=0.2, seed=42)

with tempfile.TemporaryDirectory() as tmp:
    trainer = DPOTrainer(
        model=model,
        ref_model=None,   # None -> initial weights used as reference
        args=DPOConfig(
            output_dir=tmp, num_train_epochs=1,
            per_device_train_batch_size=4, learning_rate=1e-5,
            eval_strategy='epoch', save_strategy='no', report_to='none',
            fp16=torch.cuda.is_available(),
        ),
        train_dataset=ds['train'], eval_dataset=ds['test'],
        processing_class=tok,
    )
    result = trainer.train()
    print(f'loss: {result.training_loss:.4f}')

---
## Section 8 — ORPO (Odds Ratio Preference Optimization)

Combines SFT + preference alignment in one pass. **No reference model.** Uses BASE model (not instruction-tuned).

> **Correct import:** `from trl.experimental.orpo import ORPOTrainer, ORPOConfig`  
> (`from trl import ORPOTrainer` raises `ImportError` — ORPO is not at trl's top level)

In [ ]:
import torch, tempfile
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl.experimental.orpo import ORPOTrainer, ORPOConfig  # correct import

MODEL = 'gpt2'
tok   = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL)
model.config.pad_token_id = tok.eos_token_id

data = {
    'prompt':   ['What is PEFT?', 'What is quantization?',
                 'What is a chat template?', 'What is GRPO?',
                 'Define SFT.', 'What is QLoRA?'] * 6,
    'chosen':   [
        'PEFT fine-tunes large models by training only a small subset of parameters.',
        'Quantization reduces weight precision (e.g., to 4-bit) to save memory.',
        'A chat template formats system/user/assistant turns for the model.',
        'GRPO optimises a policy using group-relative reward signals.',
        'SFT (Supervised Fine-Tuning) trains a model on labelled input-output pairs.',
        'QLoRA combines 4-bit quantization with LoRA for memory-efficient fine-tuning.',
    ] * 6,
    'rejected': [
        'PEFT makes models smaller.', 'Quantization deletes layers.',
        'A chat template is a prompt.', 'GRPO is a type of SFT.',
        'SFT trains from scratch.', 'QLoRA is just LoRA.',
    ] * 6,
}
ds = Dataset.from_dict(data).train_test_split(test_size=0.2, seed=42)

with tempfile.TemporaryDirectory() as tmp:
    trainer = ORPOTrainer(
        model=model,
        args=ORPOConfig(
            output_dir=tmp, num_train_epochs=1,
            per_device_train_batch_size=4, learning_rate=8e-6,
            eval_strategy='epoch', save_strategy='no', report_to='none',
            fp16=torch.cuda.is_available(),
        ),
        train_dataset=ds['train'], eval_dataset=ds['test'],
        processing_class=tok,
    )
    result = trainer.train()
    print(f'loss: {result.training_loss:.4f}')

---
## Section 9 — GRPO (Group Relative Policy Optimization)

Generates multiple completions per prompt, ranks by reward signal. No critic network or reference model.  
Best for: math, code, any task with **verifiable** outputs.

**Gotchas:**
- `gsm8k` has column `question` not `prompt` — rename it  
- `reward_funcs` is a **list** of callables  
- `num_generations` must be `<= per_device_train_batch_size`

In [ ]:
import torch, tempfile
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig

MODEL = 'gpt2'
tok   = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL)
model.config.pad_token_id = tok.eos_token_id

# Toy math dataset (mirrors gsm8k structure)
math_data = {
    'question': ['What is 2+2?','What is 5*3?','What is 10-4?',
                 'What is 8/2?','What is 3**2?','What is 12+7?',
                 'What is 20-11?','What is 6*6?'] * 4,
    'answer':   ['4','15','6','4','9','19','9','36'] * 4,
}
ds = Dataset.from_dict(math_data)

# KEY FIX: gsm8k uses 'question' — rename to 'prompt'
ds = ds.map(lambda x: {'prompt': x['question']}, remove_columns=['question'])
print('columns:', ds.column_names)

splits = ds.train_test_split(test_size=0.2, seed=42)

# Reward functions — each receives (prompts, completions, **kwargs) -> List[float]
def reward_correct(prompts, completions, **kwargs):
    answers = kwargs.get('answer', [''] * len(completions))
    return [1.0 if ans in comp else 0.0 for comp, ans in zip(completions, answers)]

def reward_brevity(prompts, completions, **kwargs):
    return [max(0.0, 1.0 - len(c) / 100) for c in completions]

with tempfile.TemporaryDirectory() as tmp:
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=[reward_correct, reward_brevity],  # LIST
        args=GRPOConfig(
            output_dir=tmp, num_train_epochs=1,
            per_device_train_batch_size=4,
            num_generations=4,    # <= per_device_train_batch_size
            max_completion_length=16,      # trl 1.x param name
            learning_rate=5e-6,
            eval_strategy='epoch', save_strategy='no', report_to='none',
            fp16=torch.cuda.is_available(),
        ),
        train_dataset=splits['train'],
        eval_dataset=splits['test'],
        processing_class=tok,
    )
    result = trainer.train()
    print(f'loss: {result.training_loss:.4f}')

---
## Quick Reference

| Method | Trainer | Data format | Ref model | When to use |
|--------|---------|-------------|-----------|-------------|
| SFT | `SFTTrainer` | `text` or `conversations` | — | Instruction following |
| DPO | `DPOTrainer` | `prompt, chosen, rejected` | Yes (or `None`) | Preference alignment |
| ORPO | `ORPOTrainer` | `prompt, chosen, rejected` | **None** | SFT + alignment, one pass |
| GRPO | `GRPOTrainer` | `prompt` + reward fn | **None** | Math / code / verifiable tasks |

| Save format | Method | Use case |
|-------------|--------|----------|
| LoRA adapters | `model.save_pretrained(path)` | Resume / share adapters |
| Merged 16-bit | `save_pretrained_merged(..., 'merged_16bit')` | Full model deploy |
| GGUF | `save_pretrained_gguf(..., 'q4_k_m')` | Ollama / llama.cpp |

```bash
# Run the standalone scripts
.venv/bin/python    scripts/verify_all.py          # env health check
.venv/bin/python    scripts/01_pytorch_basics.py
.venv/bin/python    scripts/04_causal_lm_finetune.py
.venv/bin/python    scripts/08_unsloth_sft.py
.venv-rl/bin/python scripts/05_dpo_example.py
.venv-rl/bin/python scripts/06_orpo_example.py
.venv-rl/bin/python scripts/07_grpo_example.py
```